Q1) Driver:
Controls the application, creates SparkContext, builds DAG, and schedules tasks.

Cluster Manager:
Allocates resources (CPU, memory) across nodes (e.g., YARN, Mesos, Standalone).

Executor:
Runs tasks on worker nodes and stores data in memory or disk.

Q2: Lazy Evaluation

Spark does not execute transformations immediately.
It builds a DAG (Directed Acyclic Graph) and executes only when an action is called.

Benifits

Optimizes execution plan
Avoids unnecessary computations
Combines multiple transformations into one stage

#Q3)

In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read CSV") \
    .getOrCreate()

In [17]:
df = spark.read.csv("source.csv", header=True, inferSchema=True)

Q4)
| **CSV**                                                | **Parquet**                                  |
| ------------------------------------------------------ | -------------------------------------------- |
| **Row-based storage**                                  | **Columnar storage**                         |
| Stores data row by row                                 | Stores data column by column                 |
| Larger file size                                       | Better compression, smaller file size        |
| Reads the entire row even if only one column is needed | Reads only the required columns              |
| Slower for analytical queries                          | Faster for analytics and big data processing |


Parquet improves performance because it:

Reads only the required columns instead of the entire dataset.
Uses efficient compression, reducing storage space and disk I/O.
Supports predicate pushdown, allowing Spark to skip unnecessary data while reading.
Processes large datasets faster than CSV.

Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [18]:
df.select("product_id", "price") \
  .filter(df.category == "Electronics")

DataFrame[product_id: int, price: int]

Q6)

In [19]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("old_name", "new_name") \
       .withColumn("price", col("price").cast("double"))

Q7)
Spark maintains a Lineage Graph (DAG) that records all the transformations performed on the data. If a worker node fails and loses its data, Spark uses this DAG to recompute only the lost partitions from the original data instead of reprocessing the entire dataset.

This provides fault tolerance because Spark can recover lost data automatically without requiring manual backups or replication of every intermediate result.

Q8)

In [20]:
df.filter(
    (df.status == "Completed") &
    (df.amount > 1000)
).show()

+----------+------------+-----------+-------+----------+---------+------+------+--------+-------+--------+
|product_id|product_name|   category|  price|base_price|   status|amount|region|priority|user_id|new_name|
+----------+------------+-----------+-------+----------+---------+------+------+--------+-------+--------+
|       101|      Laptop|Electronics|55000.0|     50000|Completed|  1200| North|    High|    1.0|  Item_A|
|       103|       Table|  Furniture| 2500.0|      2200|Completed|  3000| North|  Medium|    3.0|  Item_C|
|       104|          TV|Electronics|42000.0|     39000|Completed|  1500|  West|    High|   NULL|  Item_D|
|       106|    Keyboard|Electronics| 1500.0|      1300|Completed|  1800|  East|  Medium|    6.0|  Item_F|
|       107|        Sofa|  Furniture|18000.0|     17000|Completed|  2500| South|     Low|    7.0|  Item_G|
|       109|     Printer|Electronics| 8000.0|      7500|Completed|  2000|  West|  Medium|    9.0|  Item_I|
|       110|        Desk|  Furniture|

Q9)

Predicate Pushdown allows Spark to apply filtering conditions while reading a Parquet file instead of after loading the data.

For example:

df.filter(df.age > 30)

Spark reads only the data blocks containing rows where age > 30.

# Q10)

In [21]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

# Q11)

| Transformations            | Actions                      |
| -------------------------- | ---------------------------- |
| Lazy operations            | Trigger execution            |
| Create a new DataFrame | Return results or write data |


Transformation example

eg)
filter()

select()

Action Examples

show()
collect()
:

#Q12)

In [22]:
df.write.mode("overwrite").parquet("/content/source_parquet")

In [23]:
df = spark.read.parquet("/content/source_parquet")

In [24]:
df.filter(df.user_id.isNotNull()) \
  .write.csv(
      "/content/output",
      header=True,
      mode="overwrite"
  )

#Q13)
| **Client Mode**                                                            | **Cluster Mode**                                                                                  |
| -------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------- |
| The **Driver program** runs on the client (local machine).                 | The **Driver program** runs inside the cluster on a worker node.                                  |
| The client machine must remain connected while the application is running. | The client can disconnect after submitting the application because the cluster manages execution. |
| Suitable for development, testing, and debugging.                          | Suitable for production environments and long-running applications.                               |
| If the client machine crashes, the Spark application stops.                | If the client disconnects, the application can continue running in the cluster.                   |
| Easier to monitor logs directly from the local machine.                    | Logs are available through the cluster manager (YARN, Kubernetes, Standalone, etc.).              |


#Q14)

In [25]:
df.filter(
    (df.region == "North") |
    (df.priority == "High")
)

DataFrame[product_id: int, product_name: string, category: string, price: double, base_price: int, status: string, amount: int, region: string, priority: string, user_id: double, new_name: string, final_price: double]

#Q15)
The .show(5) method displays only the first five rows of a DataFrame, making it safe and efficient for exploring large datasets. In contrast, .collect() retrieves all rows from the DataFrame and loads them into the Driver's memory.